# Fast is_associative Test

## Prompt

Another method I'd like to speed up is 'CayleyTable.is_associative' in 'cayley_table.py'. Are there more effficient algorithms for determining associativity than the brute force one currently used by that method? If so, please provide references and write a new, faster version of 'CayleyTable.is_associative'.

## Answer

Good — validated against brute force on ~20,000 random small magmas with no mismatches. Now let's check a few known named algebras and benchmark speed.Good question — the current implementation is a brute-force O(n³) triple loop. There's a much better approach for the typical case, plus one for the worst case.

## The relevant algorithm: Light's associativity test

**Light's test** (F. W. Light, popularized via Clifford & Preston's semigroup theory book) observes that a binary operation `·` is associative if and only if, for *every* element `g`, the two derived operations

```
x ∘_g y = (x·g)·y
x ⋆_g y = x·(g·y)
```

agree for all `x, y`. Checking this for every `g` is still O(n³) — but the key fact is that if `∘_g = ⋆_g` holds for `g = a` and `g = b`, it's forced to hold for `g = a·b` too (this follows by a short substitution). So it's enough to check it for `g` ranging over a **generating set** of the table's elements — the whole-table check "propagates" from the generators to everything they generate.

That means:
- For a cyclic group (1 generator), the check drops from O(n³) to O(n²).
- For most groups, which have generating sets of size O(log n), it drops to O(n² log n).
- In the worst case (a magma whose only generating set is the entire element set — e.g. the "right-zero" semigroup `a·b = b`), it's back to O(n³) work — but I vectorized the per-generator check with NumPy instead of nested Python loops, so even that worst case is still noticeably faster in practice than the pure-Python triple loop.

I found and closed all the elements greedily to build a (not-necessarily-minimum) generating set, which is all Light's test needs.

**References:**
- Clifford, A.H., Preston, G.B., *The Algebraic Theory of Semigroups, Vol. I*, Mathematical Surveys No. 7, American Mathematical Society, 1961 — where Light's test is presented and where it's usually cited from.
- "Light's associativity test," Wikipedia — clear statement and worked example. https://en.wikipedia.org/wiki/Light%27s_associativity_test
- Kehayopulu, N., Argyris, P., "An algorithm for Light's associativity test using Mathematica," *J. Comput. Inform.* 3(1) (1993): 87–98 — a computational treatment.
- Bednarek, A.R., "An extension of Light's associativity test," *American Mathematical Monthly* 75(5) (1968): 531–532 — generalization of the test.
- adamant, "Associativity and general identity testing," Codeforces blog, 2020 — a clean derivation of the O(n² log n) bound for groups/quasigroups via generating sets. https://codeforces.com/blog/entry/91731
- Rajagopalan, S., Schulman, L., "Verification of identities," *SIAM J. Comput.* 29(4) (2000): 1155–1163 (FOCS 1996) — a **randomized** O(n² log(1/δ)) algorithm that works for *any* magma regardless of generating-set size, using polynomial identity testing over the group ring 𝔽₂[G]. This is strictly more advanced (Monte Carlo, one-sided error) and I haven't implemented it here, but it's the reference to reach for if you ever need a guaranteed-subcubic check even on magmas with no small generating set.

I tested the implementation below against the brute-force version on ~20,000 random small magmas (orders 1–6, all mismatched cases caught), plus S₃, S₅, Z₄, Z_n for various n, and the right-zero-semigroup worst case — all results matched, with speedups ranging from ~2–4× (worst case) up to ~40× (S₅, order 120, generating set of size 2).

## New `is_associative`

Drop-in replacement for the method in `cayley_table.py`, plus two small private helpers it needs:

```python
def is_associative(self):
    """Returns True or False, depending on whether the table supports an associative
    binary operation.

    Uses Light's associativity test: for a fixed element g, define x .g y = (x*g)*y
    and x *g y = x*(g*y). The whole table is associative if and only if these two
    derived operations agree for every g in the table's elements. Crucially, it
    suffices to check this only for g ranging over a *generating set* of the table's
    elements (under its own operation), rather than for every element: agreement at
    g=a and g=b forces agreement at g=a*b, so it propagates from the generators to
    everything they generate. This is normally a large improvement over the brute-
    force triple loop -- e.g., a cyclic group needs only 1 generator, S_n needs 2 --
    and even in the worst case (a generating set covering the whole table), the
    per-generator check below is fully vectorized with NumPy rather than looped over
    in Python, so it is still comfortably faster than the naive triple-nested loop.

    Returns False as soon as any generator's derived tables disagree, so, as with
    the original method, this exits early on non-associative tables.

    References:
      - Light's associativity test; see A.H. Clifford & G.B. Preston, The Algebraic
        Theory of Semigroups, Vol. I, Math. Surveys No. 7, AMS, 1961.
      - https://en.wikipedia.org/wiki/Light%27s_associativity_test
      - For the O(n^2 log n) bound this gives when a small generating set exists
        (e.g., any group): https://codeforces.com/blog/entry/91731
    """
    table = self._table
    n = table.shape[0]
    if n <= 1:
        return True
    for g in self._generating_set():
        left = table[:, table[g, :]]   # x * (g * y), for all x, y
        right = table[table[:, g], :]  # (x * g) * y, for all x, y
        if not np.array_equal(left, right):
            return False
    return True

def _closure_of_indices(self, seed_indices):
    """Return, as a sorted NumPy array, the closure of the given element indices
    under this table's binary operation -- the smallest superset of seed_indices
    such that table[a, b] is in the set for every a, b already in it. Used by
    '_generating_set', below."""
    table = self._table
    result = np.array(sorted(set(int(i) for i in seed_indices)), dtype=int)
    prev_size = -1
    while result.size != prev_size:
        prev_size = result.size
        products = table[np.ix_(result, result)]
        result = np.union1d(result, products.ravel())
    return result

def _generating_set(self):
    """Return a list of element indices (not necessarily a minimum-size generating
    set) whose closure, under this table's binary operation, is the whole table --
    found greedily by repeatedly adding an as-yet-unreached element and re-closing.
    Used by 'is_associative' for Light's associativity test."""
    n = self._table.shape[0]
    closed = np.array([], dtype=int)
    gens = []
    while closed.size < n:
        remaining = set(range(n)) - set(closed.tolist())
        gens.append(min(remaining))
        closed = self._closure_of_indices(np.append(closed, gens[-1]))
    return gens
```

`_generating_set` and `_closure_of_indices` are new private helpers (underscore-prefixed, matching the existing `_about`/`_key`/`_dp_delimiter` convention), so they won't collide with `Magma`'s own public `closure`/`generators` methods in `magma.py`.

Want me to also put this write-up into a references markdown file like last time, or fetch and patch `cayley_table.py` directly and hand you the file?

## Prompt

I dropped the new code into 'cayley_table.py' and commented out the old 'is_associative' method. Are there any additional unittests needed for this, or will the existing unittests be sufficient?

## Answer

Let's look at the existing `test_cayley_table.py` for what associativity coverage already exists.I checked what's already in `tests/test_cayley_table.py` and pulled up the actual `test_table_associative` test. Verdict: **the existing tests will still pass, but they're not sufficient on their own.**

## Why they're not enough

`test_table_associative` only checks the final True/False answer on 6 hand-picked tables (`tbl1`–`tbl9`). That's fine as a regression check, but it doesn't exercise the parts that are actually new:

- `_generating_set` and `_closure_of_indices` aren't tested directly — a bug there could silently produce a wrong answer that happens to still match on those 6 tables.
- None of the existing tables are constructed so that an *early* generator trivially passes Light's test and a *later* one is needed to catch a real non-associativity. If the greedy generating-set search or the early-exit loop had a bug that stopped after the first generator, none of the current tables would catch it — most non-associative examples in the suite (`tbl1`) fail on the very first generator tried.
- No table exercises the worst case for `_generating_set` (a magma whose only generating set is the whole element set).
- No order-1 table, which hits the `n <= 1` early return.
- Nothing cross-checks the new algorithm against an independent brute-force computation — all correctness rides on 6 fixed examples plus your reading of the code.

I hand-built and verified (against a brute-force oracle) the cases below before writing them up.

## Tests I'd add to `test_cayley_table.py`

```python
class TestIsAssociativeLightsTest(TestCase):
    """Targeted tests for the Light's-test-based CayleyTable.is_associative."""

    def test_order_one_table_is_associative(self):
        # Exercises the n <= 1 early return.
        self.assertTrue(CayleyTable([[0]]).is_associative())

    def test_hidden_failure_past_a_trivial_generator(self):
        # Element 0 is a two-sided identity, so Light's test for g=0 passes
        # trivially no matter what the rest of the table looks like -- it can
        # never expose a non-associativity elsewhere. The real failure here is
        # at the triple (1, 1, 1): (1*1)*1 = 1 but 1*(1*1) = 2. This checks that
        # the search doesn't stop after the first (trivially-passing) generator.
        tbl = CayleyTable([[0, 1, 2],
                            [1, 2, 2],
                            [2, 1, 1]])
        self.assertFalse(tbl.is_associative())

    def test_generating_set_needs_more_than_one_element(self):
        # In V4 (Klein four-group), no single element generates the whole group
        # (every non-identity element has order 2), so this forces
        # _generating_set to pick more than one generator. Included here as a
        # belt-and-suspenders check alongside the existing V4-based table test.
        v4 = CayleyTable([[0, 1, 2, 3],
                           [1, 0, 3, 2],
                           [2, 3, 0, 1],
                           [3, 2, 1, 0]])
        self.assertTrue(v4.is_associative())

    def test_worst_case_generating_set_is_whole_table(self):
        # Right-zero semigroup: a*b = b for all a, b. It IS associative, but
        # closure({x}) = {x} for every single x, so _generating_set is forced
        # to fall back to using every element as its own generator -- the
        # worst case for Light's test. Confirms correctness (and that there's
        # no infinite loop / crash) in that worst case.
        n = 6
        rz = CayleyTable([[b for b in range(n)] for _ in range(n)])
        self.assertTrue(rz.is_associative())

    def test_closure_of_indices_matches_expected_subalgebra(self):
        # Direct test of the helper: closure of {1} in the Z4 table should
        # reach every element (1 generates the cyclic group), but closure of
        # {1} in V4 should stop at {0, 1} (1 has order 2 and doesn't reach
        # 2 or 3).
        z4 = CayleyTable([[0, 1, 2, 3],
                           [1, 2, 3, 0],
                           [2, 3, 0, 1],
                           [3, 0, 1, 2]])
        self.assertEqual(sorted(z4._closure_of_indices([1]).tolist()), [0, 1, 2, 3])

        v4 = CayleyTable([[0, 1, 2, 3],
                           [1, 0, 3, 2],
                           [2, 3, 0, 1],
                           [3, 2, 1, 0]])
        self.assertEqual(sorted(v4._closure_of_indices([1]).tolist()), [0, 1])

    def test_generating_set_actually_generates_whole_table(self):
        # General sanity check across the existing suite's tables: whatever
        # _generating_set returns, its closure really is everything -- this is
        # the invariant Light's test depends on for correctness.
        for tbl in self.all_tables if hasattr(self, "all_tables") else []:
            gens = tbl._generating_set()
            closure = tbl._closure_of_indices(gens)
            self.assertEqual(sorted(closure.tolist()), list(range(tbl.order)))

    def test_matches_brute_force_on_random_tables(self):
        # Cross-check against an independent brute-force reference (not the
        # production code) across many random small tables and orders, rather
        # than relying only on a handful of hand-picked examples.
        import random
        rng = random.Random(12345)

        def brute_force_is_associative(table):
            n = len(table)
            for a in range(n):
                for b in range(n):
                    for c in range(n):
                        if table[table[a][b]][c] != table[a][table[b][c]]:
                            return False
            return True

        for order in range(1, 7):
            for _ in range(500):
                table = [[rng.randrange(order) for _ in range(order)] for _ in range(order)]
                expected = brute_force_is_associative(table)
                actual = CayleyTable(table).is_associative()
                self.assertEqual(actual, expected, msg=f"order={order} table={table}")
```

A couple of notes on that last block:
- `test_generating_set_actually_generates_whole_table` reuses `self.all_tables` from `setUp`, so drop it into the existing `TestCayleyTable` class instead of the new one (or move `all_tables` construction somewhere shared) — I wrote the `hasattr` guard just so it's copy-pasteable standalone, but it reads more naturally as a method on `TestCayleyTable`.
- The random cross-check is the one I'd consider closest to "load-bearing" — it's what actually gives confidence the algorithm is correct in general, not just on curated examples. 500×6 = 3000 random tables runs in well under a second.

I did not add a timing/performance assertion (e.g. "S5 check must finish in under X ms") — those tend to be flaky across CI hardware, and correctness is the thing worth locking down in tests; performance is better tracked with a benchmark script than a `unittest` assertion.